# Markov Maze Production Drill

Audience: you can follow the worked `markov-maze.ipynb`, but you are not yet fluent at producing the algorithm from a blank cell.

Goal for today: implement the MDP mechanics once, under time pressure, without trying to master all RL theory.

Done means:
- you can load and inspect the grid,
- you can write the transition function,
- you can run value iteration,
- you can derive a policy table,
- you can build a submission-shaped dataframe.

Time box: 45-60 minutes. Stop after the final validation cell passes.


## Mental Model

This is not classification or regression. It is a gridworld MDP.

- State: one grid cell `(row, col)`.
- Action: move up, right, down, or left.
- Transition: action changes the cell unless it hits a wall or boundary.
- Reward: small step penalty, positive goal reward.
- Value `V[state]`: how good it is to stand in that cell if you act well from there.
- Policy `policy[state]`: which action looks best from that cell.

You are practicing the production pattern, not writing a polished solution.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

MAZE_RELATIVE_PATH = Path(
    "olympiads/competition_samples/raw/romania-roai-solved/contests/algolymp/preonia-11-12-2026"
)
EXERCISE_RELATIVE_PATH = Path(
    "olympiads/IOAI Material/7. Reinforcement Learning and AI Search/exercises"
)


def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "olympiads").exists() and (path / "AGENTS.md").exists():
            return path
    return start


repo_root = find_repo_root(Path.cwd())
root_candidates = [
    Path.cwd(),
    repo_root / MAZE_RELATIVE_PATH,
    Path.cwd().parent / MAZE_RELATIVE_PATH,
    Path.cwd().parents[1] / MAZE_RELATIVE_PATH if len(Path.cwd().parents) > 1 else Path.cwd(),
]

root_path = next((path for path in root_candidates if (path / "test_data.csv").exists()), None)

if root_path is not None:
    test_df = pd.read_csv(root_path / "test_data.csv")
else:
    root_path = repo_root / EXERCISE_RELATIVE_PATH
    fixture_grid = np.zeros((6, 6), dtype=int)
    fixture_grid[0, 0] = 2
    fixture_grid[5, 5] = 3
    for wall in [(1, 1), (1, 4), (3, 2), (4, 1), (4, 4)]:
        fixture_grid[wall] = 1
    test_df = pd.DataFrame({
        "cell_index": np.arange(fixture_grid.size),
        "type": fixture_grid.ravel(),
    })
    print("test_data.csv not found; using built-in 6x6 maze fixture")

grid_size = int(np.sqrt(len(test_df)))
grid = test_df["type"].values.reshape((grid_size, grid_size))

grid


test_data.csv not found; using built-in 6x6 maze fixture


array([[2, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0],
       [0, 1, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 3]])

In [3]:
print(grid[1,1])

1


## Step 1 - Identify Special Cells

Before writing the algorithm, locate the start, goal, and walls.

Expected interpretation:
- `0`: normal cell
- `1`: wall
- `2`: start
- `3`: goal / terminal


In [4]:
# def to_tuples_with_int(mat):
#     res = []
#     print(len(mat))
#     if len(mat)==2:
#         return [int(mat[0]),  int(mat[1])]
#     for i in mat:
#         x, y = i
#         x = int(x)
#         y = int(y)
#         res.append((x, y))
#     return x

In [5]:
start = tuple(np.argwhere(grid == 2)[0])
print(tuple(np.argwhere(grid == 3)[0]))
goal = (tuple(np.argwhere(grid == 3)[0]))
walls = ( [tuple(pos) for pos in np.argwhere(grid == 1)])

print("start:", start)
print("goal:", goal)
print("walls:", walls)

assert start == (0, 0)
assert goal == (5, 5)
assert len(walls) == 5


(np.int64(5), np.int64(5))
start: (np.int64(0), np.int64(0))
goal: (np.int64(5), np.int64(5))
walls: [(np.int64(1), np.int64(1)), (np.int64(1), np.int64(4)), (np.int64(3), np.int64(2)), (np.int64(4), np.int64(1)), (np.int64(4), np.int64(4))]


## Step 2 - Write The Transition Function

Implement only the environment rule:

> If the move goes outside the grid or into a wall, stay in the same cell. Otherwise move to the new cell.

This is the MDP transition. If this is wrong, value iteration and policy extraction are meaningless.


In [6]:
ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # up, right, down, left
ACTION_NAMES = ["U", "R", "D", "L"]


def get_next_state(r, c, action):
    """Return the next (row, col) after taking one action from (r, c)."""
    dr, dc = ACTIONS[action] # 0, 1
    nr, nc = r + dr, c + dc # 1, 1

    # TODO: if (nr, nc) is outside the grid, return (r, c)
    # TODO: if grid[nr, nc] is a wall, return (r, c)
    # TODO: otherwise return (nr, nc)
    if nr>=grid_size or nr<0 or nc>=grid_size or nc<0:
        return r, c
    if  grid[nr,nc]==1:
        return r, c
    return nr, nc


In [7]:
# Transition checks. Do not move on until these pass.
assert get_next_state(0, 0, 0) == (0, 0)  # up from top boundary stays
assert get_next_state(0, 0, 1) == (0, 1)  # right from start works
assert get_next_state(0, 0, 2) == (1, 0)  # down from start works
assert get_next_state(1, 0, 1) == (1, 0)  # right into wall stays
print("transition checks passed")


transition checks passed


## Step 3 - Value Iteration

Now fill a value table.

One update means:

1. For a cell, try all 4 actions.
2. For each action, look at the next cell.
3. Score that action as immediate reward + discounted next value.
4. Store the best action score as the new cell value.

Do not update walls or the terminal goal as ordinary cells.


In [ ]:
GAMMA = 0.9
CONVERGENCE_THRESHOLD = 1e-10
REWARD_GOAL = 1.0
REWARD_STEP = -0.01

V = np.zeros((grid_size, grid_size))
policy = np.full((grid_size, grid_size), -1, dtype=int)

max_iterations = 10_000
for iteration in range(max_iterations):
    delta = 0.0
    new_V = V.copy()

    for r in range(grid_size):
        for c in range(grid_size):
            # TODO: skip walls and the goal cell
            # TODO: compute q_values for actions 0..3
            # TODO: if next cell is goal, use REWARD_GOAL
            # TODO: otherwise use REWARD_STEP + GAMMA * V[next_r, next_c]
            # TODO: set new_V[r, c] to max(q_values)
            q_values = []
            if (r,c) in walls or (r,c)==goal:
                continue
            for action, _ in enumerate(ACTIONS):
                print(r, c)
                new_pos = get_next_state(r, c, action)
                q_values.append(REWARD_GOAL if new_pos == goal else REWARD_STEP + GAMMA * V[new_pos])
            new_V[r,c] = max(q_values)
                
    delta = np.max(np.abs(new_V - V))
    V = new_V
    if delta < CONVERGENCE_THRESHOLD:
        break

print("iterations:", iteration + 1)
print(np.round(V, 3))


0 0
0 0
0 0
0 0
0 1
0 1
0 1
0 1
0 2
0 2
0 2
0 2
0 3
0 3
0 3
0 3
0 4
0 4
0 4
0 4
0 5
0 5
0 5
0 5
1 0
1 0
1 0
1 0
1 2
1 2
1 2
1 2
1 3
1 3
1 3
1 3
1 5
1 5
1 5
1 5
2 0
2 0
2 0
2 0
2 1
2 1
2 1
2 1
2 2
2 2
2 2
2 2
2 3
2 3
2 3
2 3
2 4
2 4
2 4
2 4
2 5
2 5
2 5
2 5
3 0
3 0
3 0
3 0
3 1
3 1
3 1
3 1
3 3
3 3
3 3
3 3
3 4
3 4
3 4
3 4
3 5
3 5
3 5
3 5
4 0
4 0
4 0
4 0
4 2
4 2
4 2
4 2
4 3
4 3
4 3
4 3
4 5
4 5
4 5
4 5
5 0
5 0
5 0
5 0
5 1
5 1
5 1
5 1
5 2
5 2
5 2
5 2
5 3
5 3
5 3
5 3
5 4
5 4
5 4
5 4
0 0
0 0
0 0
0 0
0 1
0 1
0 1
0 1
0 2
0 2
0 2
0 2
0 3
0 3
0 3
0 3
0 4
0 4
0 4
0 4
0 5
0 5
0 5
0 5
1 0
1 0
1 0
1 0
1 2
1 2
1 2
1 2
1 3
1 3
1 3
1 3
1 5
1 5
1 5
1 5
2 0
2 0
2 0
2 0
2 1
2 1
2 1
2 1
2 2
2 2
2 2
2 2
2 3
2 3
2 3
2 3
2 4
2 4
2 4
2 4
2 5
2 5
2 5
2 5
3 0
3 0
3 0
3 0
3 1
3 1
3 1
3 1
3 3
3 3
3 3
3 3
3 4
3 4
3 4
3 4
3 5
3 5
3 5
3 5
4 0
4 0
4 0
4 0
4 2
4 2
4 2
4 2
4 3
4 3
4 3
4 3
4 5
4 5
4 5
4 5
5 0
5 0
5 0
5 0
5 1
5 1
5 1
5 1
5 2
5 2
5 2
5 2
5 3
5 3
5 3
5 3
5 4
5 4
5 4
5 4
0 0
0 0
0 0
0 0
0 1
0 1
0 1
0 1
0 2
0 2


In [9]:
# Value-table checks. These are broad sanity checks, not exact answer leakage.
assert V.shape == (6, 6)
assert np.isfinite(V).all()
assert V[goal] == 0
assert V[start] > 0
assert V[0, 1] > V[start]  # moving closer to the goal should improve value here
print("value table checks passed")


value table checks passed


## Step 4 - Derive A Policy

The value table says how good each cell is. The policy chooses the action that leads to the best one-step outcome.

For each non-wall, non-goal cell:
- test the 4 actions,
- compute the same action score as above,
- store the `argmax` action index.


In [10]:
policy = np.full((grid_size, grid_size), -1, dtype=int)
t = []
print(grid_size)
for r in range(grid_size):
    for c in range(grid_size):
        # TODO: skip walls and goal
        # TODO: compute q_values for all actions
        # TODO: store int(np.argmax(q_values)) in policy[r, c]
        
        if (r,c) in walls or (r,c) == goal:
            continue
        q_values = []
        for action, _ in enumerate(ACTIONS):
            q_values.append(REWARD_STEP + GAMMA * V[get_next_state(r,c,action)])
        t.append(np.argmax(q_values))
        policy[r,c] = int(np.argmax(q_values))

print("policy action ids:")
print(policy)


print("policy action names:")
policy_names = np.full(policy.shape, "#", dtype=object)
for r in range(grid_size):
    for c in range(grid_size):
        if policy[r, c] >= 0:
            policy_names[r, c] = ACTION_NAMES[policy[r, c]]
        elif grid[r, c] == 3:
            policy_names[r, c] = "G"
print(policy_names)


6
policy action ids:
[[ 1  1  1  1  1  2]
 [ 2 -1  1  2 -1  2]
 [ 1  1  1  1  1  2]
 [ 2  0 -1  1  1  2]
 [ 2 -1  1  2 -1  1]
 [ 1  1  1  1  0 -1]]
policy action names:
[['R' 'R' 'R' 'R' 'R' 'D']
 ['D' '#' 'R' 'D' '#' 'D']
 ['R' 'R' 'R' 'R' 'R' 'D']
 ['D' 'U' '#' 'R' 'R' 'D']
 ['D' '#' 'R' 'D' '#' 'R']
 ['R' 'R' 'R' 'R' 'U' 'G']]


In [11]:
# Policy checks.
assert policy.shape == (6, 6)
print(policy[start])
assert policy[start] in [1, 2]  # from top-left, right or down are plausible first moves
assert policy[goal] == -1
for wall in walls:
    assert policy[wall] == -1
print("policy checks passed")


1
policy checks passed


## Step 5 - Build Submission-Shaped Output

This final part is competition plumbing.

Subtask 1 wants one value per cell. Subtask 2 wants one action per non-wall, non-goal cell.


In [12]:
subtask1 = []
for r in range(grid_size):
    for c in range(grid_size):
        subtask1.append(float(V[r, c]))

subtask2 = []
for r in range(grid_size):
    for c in range(grid_size):
        if grid[r, c] in [1, 3]:
            continue
        subtask2.append(int(policy[r, c]))


def build_subtask(sid, answers):
    ids = (
        test_df["cell_index"]
        if sid == 1
        else test_df[(test_df["type"] != 1) & (test_df["type"] != 3)]["cell_index"]
    )
    return pd.DataFrame({"subtaskID": sid, "datapointID": ids, "answer": answers})

submission = pd.concat([
    build_subtask(1, subtask1),
    build_subtask(2, subtask2),
], ignore_index=True)

submission.head()


,subtaskID,datapointID,answer
0,1,0,0.326163
1,1,1,0.373514
2,1,2,0.426127
3,1,3,0.484585
4,1,4,0.549539


In [13]:
assert len(subtask1) == 36
assert len(subtask2) == int(((grid != 1) & (grid != 3)).sum())
assert list(submission.columns) == ["subtaskID", "datapointID", "answer"]
assert submission["subtaskID"].value_counts().to_dict() == {1: 36, 2: len(subtask2)}

out_path = root_path / "submission_drill.csv"
submission.to_csv(out_path, index=False)
print("wrote", out_path)


wrote c:\Users\Lefteris Dragasakis\Documents\GitHub\Supervised-Learning-Experiments\olympiads\IOAI Material\7. Reinforcement Learning and AI Search\exercises\submission_drill.csv


## Stop Condition

You are done when:

- transition checks pass,
- value table checks pass,
- policy checks pass,
- `submission_drill.csv` is written.

What you should be able to say after this:

> I can turn a grid into states/actions/transitions, run value iteration, derive a greedy policy, and export the required table.

Do not polish this notebook. If all checks pass, move on.


## What I learned

- A gridworld MDP uses cells as states and moves as actions.
- `get_next_state` defines the transition rule.
- Value iteration repeatedly estimates how good each cell is.
- The policy chooses the action with the highest numeric action value.
- `argmax` must be used on scores, not coordinates.